# Autenticação Facial — Camada complementar à senha

**Contexto.** O setor de fraudes identificou contratações indevidas (ex.: crédito
pessoal) em que a **senha foi validada corretamente**, mas o cliente nega ter
contratado o serviço. A senha sozinha não prova *presença* nem *identidade física*
de quem opera. Este notebook implementa uma **camada de autenticação facial
complementar**, com três capacidades:

1. **Detecção de faces** — localizar o rosto no quadro.
2. **Identificação de faces** — confirmar que o rosto é do cliente cadastrado.
3. **Liveness (vivacidade)** — garantir que é uma pessoa real ao vivo, e não uma
   **foto estática**.

Quando o cliente **não é autenticado**, o fluxo o encaminha para uma **esteira
dedicada** e grava **evidências** (frame, scores, limiares e motivo) em
`evidences/` para a **área de IA** recalibrar parâmetros e limiares.

**Stack:** OpenCV · MediaPipe **Tasks API** (FaceDetector + FaceLandmarker) ·
DeepFace/ArcFace · scikit-learn · matplotlib.

> Execute as células **em ordem**. As chamadas que abrem a **webcam** estão
> comentadas — descomente para rodar ao vivo.

## 0. Setup & Imports

In [ ]:
# Descomente na PRIMEIRA execucao para instalar as dependencias:
# %pip install -r requirements.txt

In [1]:
import os, json, pickle, time, csv, urllib.request
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

print("OpenCV   :", cv2.__version__)
print("MediaPipe:", mp.__version__)

OpenCV   : 4.13.0
MediaPipe: 0.10.35


## 1. Configuração central

Todos os parâmetros e limiares ficam aqui, num único lugar, para facilitar o
ajuste pela área de IA.

| Parâmetro | Significado |
|---|---|
| `THRESHOLD_SIMILARIDADE` | corte da similaridade de cosseno p/ aceitar a identidade |
| `EAR_THRESHOLD` | abaixo deste valor o olho é considerado fechado (piscada) |
| `BLINK_THRESHOLD` | score de *blendshape* de olho fechado (sinal alternativo de piscada) |
| `PISCADAS_NECESSARIAS` | nº de piscadas exigidas no desafio de liveness |
| `N_FOTOS_CADASTRO` | nº de frames capturados no cadastro |
| `MODELO` | modelo de embedding do DeepFace (padrão `ArcFace`) |

> Referência: para ArcFace, a *distância* de cosseno de corte do DeepFace é ~0.68
> (⇒ similaridade ~0.32). Usamos `0.40` como padrão um pouco mais rígido —
> ajuste conforme o relatório FAR×FRR da última seção.

In [2]:
BASE_DIR   = Path.cwd()                 # diretorio do notebook
DATA_DIR   = BASE_DIR / "data"
ENROLL_DIR = DATA_DIR / "enrolled"
EMB_DIR    = DATA_DIR / "embeddings"
EVID_DIR   = BASE_DIR / "evidences"
MODELS_DIR = BASE_DIR / "models"
for d in (ENROLL_DIR, EMB_DIR, EVID_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "MODELO": "ArcFace",            # modelo de embedding do DeepFace
    "DETECTOR_BACKEND": "opencv",   # backend de deteccao usado pelo DeepFace
    "THRESHOLD_SIMILARIDADE": 0.40, # corte de cosine similarity p/ aceitar identidade
    "EAR_THRESHOLD": 0.21,          # abaixo disso = olho fechado (via EAR)
    "BLINK_THRESHOLD": 0.50,        # acima disso = olho fechado (via blendshape)
    "EAR_CONSEC_FRAMES": 2,         # frames consecutivos fechados p/ contar 1 piscada
    "PISCADAS_NECESSARIAS": 2,      # piscadas exigidas no desafio
    "LIVENESS_TIMEOUT": 15,         # segundos p/ completar o desafio
    "N_FOTOS_CADASTRO": 5,          # frames capturados no cadastro
    "BLUR_MIN": 60.0,               # variancia minima do Laplaciano (anti-borrao)
    "CAM_INDEX": 0,                 # indice da webcam
}
CONFIG

{'MODELO': 'ArcFace',
 'DETECTOR_BACKEND': 'opencv',
 'THRESHOLD_SIMILARIDADE': 0.4,
 'EAR_THRESHOLD': 0.21,
 'BLINK_THRESHOLD': 0.5,
 'EAR_CONSEC_FRAMES': 2,
 'PISCADAS_NECESSARIAS': 2,
 'LIVENESS_TIMEOUT': 15,
 'N_FOTOS_CADASTRO': 5,
 'BLUR_MIN': 60.0,
 'CAM_INDEX': 0}

## 2. Modelos do MediaPipe (Tasks API)

A versão instalada do MediaPipe usa a **Tasks API**, que carrega modelos a partir
de arquivos. Baixamos uma vez para `models/`:

* `blaze_face_short_range.tflite` — **detecção** de faces.
* `face_landmarker.task` — **malha facial** (478 pontos) + *blendshapes* (usados
  no liveness por piscada).

In [3]:
_MODELOS = {
    "blaze_face_short_range.tflite":
        "https://storage.googleapis.com/mediapipe-models/face_detector/"
        "blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
    "face_landmarker.task":
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
        "face_landmarker/float16/1/face_landmarker.task",
}

def baixar_modelos():
    for nome, url in _MODELOS.items():
        dest = MODELS_DIR / nome
        if dest.exists() and dest.stat().st_size > 0:
            continue
        print("baixando", nome, "...")
        urllib.request.urlretrieve(url, dest)
    print("Modelos prontos em", MODELS_DIR)

baixar_modelos()

Modelos prontos em c:\Users\stgab\OneDrive\Documentos\GitHub\Grupo_2_atividades\Autenticacao_facial_claude\models


## 3. Utilitários

Funções de apoio: conversão p/ imagem do MediaPipe, medida de nitidez
(anti-borrão), exibição de frames no notebook e desenho de caixa delimitadora.

In [4]:
def to_mp_image(img_bgr):
    """Converte um frame BGR (OpenCV) em mp.Image (SRGB) para a Tasks API."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))

def variancia_laplaciano(img_bgr):
    """Mede nitidez: valores altos = nitido; baixos = borrado."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def mostrar(img_bgr, titulo=""):
    """Exibe um frame BGR no notebook via matplotlib."""
    plt.figure(figsize=(5, 5))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(titulo); plt.axis("off"); plt.show()

def desenhar_caixa(img_bgr, bbox, cor=(0, 255, 0), label=None):
    x, y, w, h = bbox
    out = img_bgr.copy()
    cv2.rectangle(out, (x, y), (x + w, y + h), cor, 2)
    if label:
        cv2.putText(out, label, (x, max(0, y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2)
    return out

## 4. Detector de Faces

Usamos o **FaceDetector** da Tasks API. A função retorna as caixas
delimitadoras `(x, y, w, h)`; o fluxo de cadastro exige **exatamente 1 rosto**.

In [5]:
_FD_PATH = str(MODELS_DIR / "blaze_face_short_range.tflite")

def _criar_detector(min_conf=0.6):
    opts = vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path=_FD_PATH),
        running_mode=vision.RunningMode.IMAGE,
        min_detection_confidence=min_conf,
    )
    return vision.FaceDetector.create_from_options(opts)

_FACE_DETECTOR = _criar_detector()

def detectar_faces(img_bgr):
    """Retorna lista de bboxes (x, y, w, h) via MediaPipe Tasks FaceDetector."""
    res = _FACE_DETECTOR.detect(to_mp_image(img_bgr))
    faces = []
    for d in res.detections:
        bb = d.bounding_box
        faces.append((max(0, bb.origin_x), max(0, bb.origin_y), bb.width, bb.height))
    return faces

def recortar_face(img_bgr, bbox, margem=0.2):
    x, y, w, h = bbox
    mx, my = int(w * margem), int(h * margem)
    H, W = img_bgr.shape[:2]
    x0, y0 = max(0, x - mx), max(0, y - my)
    x1, y1 = min(W, x + w + mx), min(H, y + h + my)
    return img_bgr[y0:y1, x0:x1]

## 5. Liveness ativo (piscada)

Para impedir que um fraudador use uma **foto estática**, exigimos um desafio:
o usuário precisa **piscar** algumas vezes diante da câmera.

Usamos o **FaceLandmarker** (478 pontos + *blendshapes*). Detectamos o olho
fechado por **dois sinais complementares**:

* **EAR (Eye Aspect Ratio)** — calculado a partir dos landmarks:

$$EAR = \frac{\lVert p_2 - p_6 \rVert + \lVert p_3 - p_5 \rVert}{2\,\lVert p_1 - p_4 \rVert}$$

* **Blendshape** `eyeBlink` — score do próprio modelo para "olho fechado".

Olho considerado fechado se `EAR < EAR_THRESHOLD` **ou** `blink > BLINK_THRESHOLD`.
Uma queda por alguns frames consecutivos conta como **1 piscada**; uma foto
parada **não pisca** → reprovada.

In [6]:
_FL_PATH = str(MODELS_DIR / "face_landmarker.task")

# indices dos olhos no FaceMesh, na ordem [canto1, sup1, sup2, canto2, inf2, inf1]
OLHO_ESQ = [33, 160, 158, 133, 153, 144]
OLHO_DIR = [362, 385, 387, 263, 373, 380]

def _criar_landmarker():
    opts = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=_FL_PATH),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=True,
    )
    return vision.FaceLandmarker.create_from_options(opts)

_FACE_LANDMARKER = _criar_landmarker()

def _ear(pts):
    p1, p2, p3, p4, p5, p6 = [np.array(p) for p in pts]
    vert  = np.linalg.norm(p2 - p6) + np.linalg.norm(p3 - p5)
    horiz = 2.0 * np.linalg.norm(p1 - p4)
    return float(vert / horiz) if horiz > 0 else 0.0

def calcular_ear(landmarks, w, h):
    """EAR medio dos dois olhos a partir dos landmarks (normalizados) do FaceMesh."""
    def coords(idxs):
        return [(landmarks[i].x * w, landmarks[i].y * h) for i in idxs]
    return (_ear(coords(OLHO_ESQ)) + _ear(coords(OLHO_DIR))) / 2.0

def analisar_frame(img_bgr):
    """Retorna {'ear':..., 'blink':...} do rosto no frame, ou None se nao houver rosto."""
    res = _FACE_LANDMARKER.detect(to_mp_image(img_bgr))
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]
    h, w = img_bgr.shape[:2]
    blink = None
    if res.face_blendshapes:
        d = {c.category_name: c.score for c in res.face_blendshapes[0]}
        blink = (d.get("eyeBlinkLeft", 0.0) + d.get("eyeBlinkRight", 0.0)) / 2.0
    return {"ear": calcular_ear(lm, w, h), "blink": blink}

In [7]:
def _olho_fechado(m):
    if m is None:
        return False
    if m["ear"] < CONFIG["EAR_THRESHOLD"]:
        return True
    return m["blink"] is not None and m["blink"] > CONFIG["BLINK_THRESHOLD"]

def desafio_liveness(cam=None, mostrar_janela=True):
    """Abre a webcam e exige PISCADAS_NECESSARIAS piscadas dentro do timeout.
    Retorna (aprovado: bool, resumo: dict, ultimo_frame)."""
    own = cam is None
    if own:
        cam = cv2.VideoCapture(CONFIG["CAM_INDEX"])
    piscadas = 0
    contador_fechado = 0
    ear_min = 1.0
    t0 = time.time()
    ultimo_frame = None
    while True:
        ok, frame = cam.read()
        if not ok:
            break
        ultimo_frame = frame.copy()
        m = analisar_frame(frame)
        if m is not None:
            ear_min = min(ear_min, m["ear"])
        if _olho_fechado(m):
            contador_fechado += 1
        else:
            if contador_fechado >= CONFIG["EAR_CONSEC_FRAMES"]:
                piscadas += 1
            contador_fechado = 0
        restante = CONFIG["LIVENESS_TIMEOUT"] - (time.time() - t0)
        if mostrar_janela:
            disp = frame.copy()
            txt = (f"Pisque {CONFIG['PISCADAS_NECESSARIAS']}x | piscadas: {piscadas}"
                   f" | EAR: {m['ear']:.2f}") if m is not None else "Rosto nao detectado"
            cv2.putText(disp, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(disp, f"Tempo: {max(0, restante):.0f}s", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            cv2.imshow("Liveness - pisque (ESC sai)", disp)
            if cv2.waitKey(1) & 0xFF == 27:
                break
        if piscadas >= CONFIG["PISCADAS_NECESSARIAS"] or restante <= 0:
            break
    if own:
        cam.release(); cv2.destroyAllWindows()
    aprovado = piscadas >= CONFIG["PISCADAS_NECESSARIAS"]
    resumo = {"piscadas": piscadas, "piscadas_exigidas": CONFIG["PISCADAS_NECESSARIAS"],
              "ear_min": round(float(ear_min), 3), "tempo_s": round(time.time() - t0, 1)}
    return aprovado, resumo, ultimo_frame

## 6. Identificação de faces (DeepFace / ArcFace)

Extraímos um **embedding** (vetor que representa o rosto) com o modelo
**ArcFace** do DeepFace e comparamos por **similaridade de cosseno** com os
embeddings cadastrados do `user_id` alegado.

> O primeiro `import` do DeepFace baixa os pesos do modelo (alguns MB) — pode
> demorar na primeira vez.

In [8]:
from deepface import DeepFace

def extrair_embedding(img_bgr, enforce=True):
    """Extrai o vetor de embedding (ArcFace) de um frame BGR. Retorna np.array ou None."""
    try:
        reps = DeepFace.represent(
            img_path=img_bgr,                     # numpy BGR (padrao do OpenCV)
            model_name=CONFIG["MODELO"],
            detector_backend=CONFIG["DETECTOR_BACKEND"],
            enforce_detection=enforce,
        )
    except Exception as e:
        print("Falha ao extrair embedding:", e)
        return None
    if not reps:
        return None
    return np.array(reps[0]["embedding"], dtype=np.float32)

def similaridade(emb_a, emb_b):
    return float(cosine_similarity(emb_a.reshape(1, -1), emb_b.reshape(1, -1))[0, 0])

In [9]:
def _emb_path(user_id):
    return EMB_DIR / f"{user_id}.pkl"

def salvar_cadastro(user_id, embeddings):
    with open(_emb_path(user_id), "wb") as f:
        pickle.dump({"user_id": user_id, "modelo": CONFIG["MODELO"],
                     "embeddings": [e.tolist() for e in embeddings]}, f)

def carregar_cadastro(user_id):
    p = _emb_path(user_id)
    if not p.exists():
        return None
    with open(p, "rb") as f:
        d = pickle.load(f)
    return [np.array(e, dtype=np.float32) for e in d["embeddings"]]

def usuarios_cadastrados():
    return [p.stem for p in EMB_DIR.glob("*.pkl")]

def verificar_identidade(emb_live, user_id):
    """Compara o embedding ao vivo com os cadastrados do user_id.
    Retorna (aprovado: bool, score_max: float)."""
    cadastrados = carregar_cadastro(user_id)
    if not cadastrados:
        return False, 0.0
    smax = max(similaridade(emb_live, e) for e in cadastrados)
    return smax >= CONFIG["THRESHOLD_SIMILARIDADE"], smax

## 7. Esteira dedicada & Evidências

Em **qualquer reprovação** (liveness, rosto não corresponde, baixa qualidade) o
cliente é encaminhado à **esteira dedicada** e gravamos a evidência: o frame
(`.jpg`), um `.json` com metadados e uma linha em `evidences/log.csv`. É esse
material que a **área de IA** usa para ajustar limiares e aperfeiçoar o modelo.

In [10]:
LOG_CSV = EVID_DIR / "log.csv"

def registrar_evidencia(user_id, etapa, motivo, score, limiar, frame=None, extra=None):
    """Salva frame + JSON + linha no CSV para a area de IA analisar."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    nome = f"{user_id}_{etapa}_{ts}"
    img_path = ""
    if frame is not None:
        img_path = str(EVID_DIR / f"{nome}.jpg")
        cv2.imwrite(img_path, frame)
    registro = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "user_id": user_id, "etapa": etapa, "motivo": motivo,
        "score": round(float(score), 4), "limiar": float(limiar),
        "imagem": img_path, "extra": extra or {},
    }
    with open(EVID_DIR / f"{nome}.json", "w", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)
    novo = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="", encoding="utf-8") as f:
        wcsv = csv.writer(f)
        if novo:
            wcsv.writerow(["timestamp", "user_id", "etapa", "motivo", "score", "limiar", "imagem"])
        wcsv.writerow([registro["timestamp"], user_id, etapa, motivo,
                       registro["score"], limiar, img_path])
    print(f"[ESTEIRA DEDICADA] Evidencia registrada: {nome}")
    return registro

## 8. Fluxo de Cadastro (Enrollment)

Captura `N_FOTOS_CADASTRO` frames pela webcam. Cada captura passa por um
**quality gate** (exatamente 1 rosto + nitidez mínima) antes de virar embedding.
Guardamos **vários embeddings por usuário** → identificação mais robusta.

**Controles:** `ESPAÇO` captura · `ESC` encerra.

In [11]:
def cadastrar_usuario(user_id):
    cam = cv2.VideoCapture(CONFIG["CAM_INDEX"])
    capturados = []
    pasta_fotos = ENROLL_DIR / user_id
    pasta_fotos.mkdir(parents=True, exist_ok=True)
    print(f"Cadastro de '{user_id}': ESPACO=capturar ({CONFIG['N_FOTOS_CADASTRO']} fotos), ESC=sair.")
    while len(capturados) < CONFIG["N_FOTOS_CADASTRO"]:
        ok, frame = cam.read()
        if not ok:
            break
        faces = detectar_faces(frame)
        disp = frame.copy()
        cor = (0, 255, 0) if len(faces) == 1 else (0, 0, 255)
        for bb in faces:
            disp = desenhar_caixa(disp, bb, cor)
        cv2.putText(disp, f"Capturadas: {len(capturados)}/{CONFIG['N_FOTOS_CADASTRO']}  (ESPACO=capturar)",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow("Cadastro - ESPACO captura / ESC sai", disp)
        k = cv2.waitKey(1) & 0xFF
        if k == 27:               # ESC
            break
        if k == 32:               # ESPACO
            if len(faces) != 1:
                print("  -> precisa de exatamente 1 rosto no quadro."); continue
            nitidez = variancia_laplaciano(recortar_face(frame, faces[0]))
            if nitidez < CONFIG["BLUR_MIN"]:
                print(f"  -> imagem borrada (nitidez={nitidez:.0f}). Tente de novo."); continue
            emb = extrair_embedding(frame)
            if emb is None:
                print("  -> nao consegui extrair embedding."); continue
            capturados.append(emb)
            cv2.imwrite(str(pasta_fotos / f"{len(capturados)}.jpg"), frame)
            print(f"  -> foto {len(capturados)} OK (nitidez={nitidez:.0f}).")
    cam.release(); cv2.destroyAllWindows()
    if capturados:
        salvar_cadastro(user_id, capturados)
        print(f"Cadastro concluido: {len(capturados)} embeddings em {_emb_path(user_id)}")
    else:
        print("Nenhuma foto capturada; cadastro nao salvo.")
    return capturados

In [12]:
# >>> Defina o ID do cliente e rode o cadastro olhando para a webcam:
USER_ID = "cliente_001"
# cadastrar_usuario(USER_ID)        # <- descomente p/ executar (abre a webcam)

print("Usuarios cadastrados:", usuarios_cadastrados())

Usuarios cadastrados: []


## 9. Fluxo de Autenticação

Fluxo completo e na ordem correta de segurança:

1. **Liveness ativo** (pisque) — só prossegue se for uma pessoa **viva**.
2. **Identificação** — compara o rosto ao vivo com o cadastro do `user_id` alegado.
3. **Decisão** — `AUTENTICADO` ou, em caso de falha, **esteira dedicada** + evidência.

In [13]:
def autenticar(user_id_alegado):
    """Liveness ativo -> identificacao -> decisao. Reprovacao gera evidencia."""
    if carregar_cadastro(user_id_alegado) is None:
        print(f"Usuario '{user_id_alegado}' nao cadastrado.")
        return {"autenticado": False, "motivo": "nao_cadastrado"}

    print(">> Etapa 1/2: Deteccao de vivacidade (pisque os olhos)...")
    vivo, info_liv, frame = desafio_liveness()
    if not vivo:
        registrar_evidencia(user_id_alegado, "liveness", "liveness_reprovado",
                            score=info_liv["piscadas"], limiar=CONFIG["PISCADAS_NECESSARIAS"],
                            frame=frame, extra=info_liv)
        print(f"REPROVADO no liveness {info_liv}. -> Esteira dedicada.")
        return {"autenticado": False, "motivo": "liveness", "info": info_liv}
    print(f"   Liveness OK: {info_liv}")

    print(">> Etapa 2/2: Identificacao facial...")
    emb = extrair_embedding(frame) if frame is not None else None
    if emb is None:
        registrar_evidencia(user_id_alegado, "identificacao", "sem_rosto_valido",
                            score=0.0, limiar=CONFIG["THRESHOLD_SIMILARIDADE"], frame=frame)
        print("Nao foi possivel extrair rosto valido. -> Esteira dedicada.")
        return {"autenticado": False, "motivo": "sem_rosto"}

    ok, score = verificar_identidade(emb, user_id_alegado)
    if ok:
        print(f"AUTENTICADO  (similaridade={score:.3f} >= {CONFIG['THRESHOLD_SIMILARIDADE']})")
        return {"autenticado": True, "score": score, "liveness": info_liv}
    registrar_evidencia(user_id_alegado, "identificacao", "rosto_nao_corresponde",
                        score=score, limiar=CONFIG["THRESHOLD_SIMILARIDADE"], frame=frame,
                        extra=info_liv)
    print(f"REPROVADO  (similaridade={score:.3f} < {CONFIG['THRESHOLD_SIMILARIDADE']}). -> Esteira dedicada.")
    return {"autenticado": False, "motivo": "identificacao", "score": score}

In [ ]:
# resultado = autenticar("cliente_001")   # <- descomente p/ executar (abre a webcam)
# resultado

## 10. (Opcional) Liveness passivo — anti-spoofing por imagem

O DeepFace traz um modelo **Silent-Face** de anti-spoofing que avalia a textura
de **uma única imagem** (sem exigir piscada). É um sinal **complementar** ao
liveness ativo — útil como segunda barreira e como mais material para a área de
IA. Combinação recomendada: exigir `liveness_ativo == True` **e**
`is_real == True`.

In [14]:
def checar_spoof_passivo(img_bgr):
    """Liveness PASSIVO (opcional) via Silent-Face do DeepFace.
    Retorna (is_real: bool|None, score: float|None)."""
    try:
        faces = DeepFace.extract_faces(img_bgr, detector_backend=CONFIG["DETECTOR_BACKEND"],
                                       anti_spoofing=True, enforce_detection=False)
    except Exception as e:
        print("anti-spoof indisponivel:", e)
        return None, None
    if not faces:
        return None, None
    f0 = faces[0]
    return bool(f0.get("is_real", False)), float(f0.get("antispoof_score", 0.0))

## 11. Relatório & Ajuste de limiares (FAR × FRR)

Duas visões para a **área de IA**:

* `relatorio_evidencias()` — resume as não-identificações registradas.
* `analisar_limiares()` — usando os usuários já cadastrados, estima as
  distribuições de similaridade **genuíno** (mesma pessoa) × **impostor**
  (pessoas diferentes) e mostra onde cai o limiar atual. Mover o limiar para a
  direita reduz **falsos aceites (FAR)** e aumenta **falsos rejeites (FRR)**, e
  vice-versa.

In [15]:
import pandas as pd

def relatorio_evidencias():
    if not LOG_CSV.exists():
        print("Sem evidencias registradas ainda.")
        return None
    df = pd.read_csv(LOG_CSV)
    print(f"Total de evidencias: {len(df)}")
    print(df["motivo"].value_counts())
    return df

def analisar_limiares():
    """Distribuicoes genuino x impostor entre os usuarios cadastrados."""
    users = usuarios_cadastrados()
    embs = {u: carregar_cadastro(u) for u in users}
    genuino, impostor = [], []
    for u in users:
        es = embs[u] or []
        for i in range(len(es)):
            for j in range(i + 1, len(es)):
                genuino.append(similaridade(es[i], es[j]))
    for a_i in range(len(users)):
        for b_i in range(a_i + 1, len(users)):
            for a in (embs[users[a_i]] or []):
                for b in (embs[users[b_i]] or []):
                    impostor.append(similaridade(a, b))
    plt.figure(figsize=(7, 4))
    if genuino:  plt.hist(genuino,  bins=20, alpha=0.6, label="genuino (mesma pessoa)")
    if impostor: plt.hist(impostor, bins=20, alpha=0.6, label="impostor (pessoas diferentes)")
    plt.axvline(CONFIG["THRESHOLD_SIMILARIDADE"], color="red", ls="--", label="limiar atual")
    plt.xlabel("similaridade de cosseno"); plt.ylabel("frequencia"); plt.legend()
    plt.title("Distribuicao de similaridades - ajuste do limiar (FAR x FRR)")
    plt.show()
    return {"genuino": genuino, "impostor": impostor}

# relatorio_evidencias()
# analisar_limiares()

## 12. (Opcional) Modo offline — reprodutibilidade sem webcam

Para apresentação/correção sem câmera, é possível cadastrar e autenticar a
partir de **arquivos de imagem**. ⚠️ Este modo **não executa o liveness ativo**
(que precisa de vídeo/webcam) — serve para validar **detecção + identificação**.

In [16]:
def cadastrar_por_arquivos(user_id, caminhos):
    embs = []
    for c in caminhos:
        img = cv2.imread(str(c))
        if img is None:
            print("nao consegui ler:", c); continue
        emb = extrair_embedding(img)
        if emb is not None:
            embs.append(emb)
    if embs:
        salvar_cadastro(user_id, embs)
        print(f"{len(embs)} embeddings salvos p/ '{user_id}'")
    return embs

def autenticar_por_arquivo(user_id_alegado, caminho_img):
    """Versao offline (sem webcam): apenas identificacao a partir de uma imagem."""
    if carregar_cadastro(user_id_alegado) is None:
        print("usuario nao cadastrado"); return None
    img = cv2.imread(str(caminho_img))
    if img is None:
        print("nao consegui ler a imagem"); return None
    emb = extrair_embedding(img)
    if emb is None:
        print("rosto nao detectado"); return None
    ok, score = verificar_identidade(emb, user_id_alegado)
    print(("AUTENTICADO" if ok else "REPROVADO") +
          f" (sim={score:.3f}, limiar={CONFIG['THRESHOLD_SIMILARIDADE']})")
    return {"autenticado": ok, "score": score}

# Exemplo:
# cadastrar_por_arquivos("cliente_teste", ["data/enrolled/cliente_teste/1.jpg"])
# autenticar_por_arquivo("cliente_teste", "alguma_foto.jpg")

## Resumo do fluxo

```
                 +------------------+
   webcam  --->  | Deteccao de face |
                 +------------------+
                          |
                          v
                 +------------------+      reprovado     +-------------------+
                 | Liveness (pisca) |  ---------------->  | Esteira dedicada  |
                 +------------------+                     |  + evidencia.json |
                          | vivo                          |  + frame.jpg      |
                          v                               |  + log.csv        |
                 +------------------+      < limiar        +---------+---------+
                 | Identificacao    |  ----------------------------->|
                 | (ArcFace cosine) |                                | area de IA
                 +------------------+                                | recalibra
                          | >= limiar                                v limiares
                          v
                   AUTENTICADO
```

A **senha** continua válida; a face é a **segunda camada**. Toda não-identificação
vira **evidência rotulada** que alimenta o ciclo de melhoria do modelo —
exatamente o que o setor de fraudes pediu.